# LCEL Deep Dive — LangChain Expression Language

In Tutorial 01, we used the `|` pipe operator to build simple chains like `prompt | llm | parser`.

But real-world apps need more:
- What if you need the **original input AND the LLM output** together?
- What if you want to run **3 chains at the same time** on the same input?
- What if you need to **route** a question to different chains based on its type?
- What if your **primary model goes down** and you need a backup?

LCEL solves all of this with 5 building blocks. Let's go through each one.

---

In [ ]:
!pip install -q langchain langchain-openai langchain-anthropic

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,   # forwards input without changing it
    RunnableParallel,      # runs multiple chains at the same time
    RunnableLambda,        # wraps any Python function as a chain step
    RunnableBranch,        # routes input to different chains (like if/else)
)

# We'll use GPT-4o-mini as our default model throughout
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

---

## 1 · RunnablePassthrough — Keep Original Input While Adding New Data

### The Problem
Imagine you send `{"topic": "RAG"}` into a chain. The chain summarizes it and returns a string.
But now you've **lost** the original topic — all you have is the summary text.

### The Solution
`RunnablePassthrough` forwards your input through unchanged. Combined with `.assign()`,
it lets you **keep the original data AND add new computed fields** to it.

Think of it like a conveyor belt — your original package stays on the belt,
and `.assign()` sticks a new label on it as it passes through.

In [ ]:
# --- Basic Example: Passthrough just forwards the input ---

passthrough = RunnablePassthrough()

# Whatever you put in, you get out unchanged
result = passthrough.invoke({"topic": "RAG"})
print(f"Input forwarded as-is: {result}")  
# Output: {"topic": "RAG"} — nothing changed

In [ ]:
# --- Real Use Case: Keep the original topic AND add a summary ---

# Step 1: Build a simple summarization chain
prompt = ChatPromptTemplate.from_template("Summarize this in 1 sentence: {topic}")
summary_chain = prompt | llm | StrOutputParser()

# Step 2: Use .assign() to ADD the summary while KEEPING the original input
#
# Without .assign(): you'd only get the summary string, losing {topic}
# With .assign():    you get {topic: "...", summary: "..."}

chain_with_passthrough = RunnablePassthrough.assign(
    summary=summary_chain  # run summary_chain and store result under "summary" key
)

result = chain_with_passthrough.invoke({"topic": "Transformer architecture"})

# Now we have BOTH the original topic and the generated summary
print(f"Original topic (preserved): {result['topic']}")
print(f"Generated summary (added):  {result['summary']}")

### ✅ What Just Happened?

1. We passed `{"topic": "Transformer architecture"}` into the chain
2. `RunnablePassthrough.assign()` did two things simultaneously:
   - **Forwarded** `{"topic": "Transformer architecture"}` unchanged
   - **Ran** `summary_chain` on that input and stored the result under a new `"summary"` key
3. The output is a dict with both: `{"topic": "...", "summary": "..."}`

**When to use this:** Anytime you need the original input downstream — for example, in RAG pipelines where you need both the user's question AND the retrieved context.

---

## 2 · RunnableParallel — Run Multiple Chains at the Same Time

### The Problem
You want to analyze a technology from 3 angles: pros, cons, and use cases.
Running them one after another is slow — each waits for the previous to finish.

### The Solution
`RunnableParallel` runs all 3 chains **simultaneously** on the same input.
It's like having 3 analysts working on the same question at the same time
instead of passing it one by one.

In [ ]:
# Define 3 different prompts — each analyzes from a different angle
pros_prompt = ChatPromptTemplate.from_template(
    "List 3 pros of {technology}. Be concise — one line each."
)
cons_prompt = ChatPromptTemplate.from_template(
    "List 3 cons of {technology}. Be concise — one line each."
)
usecase_prompt = ChatPromptTemplate.from_template(
    "Give 3 real-world use cases for {technology}. Be concise — one line each."
)

# RunnableParallel: each key runs its own chain SIMULTANEOUSLY
# All 3 chains receive the SAME input: {"technology": "LangChain"}
# All 3 run AT THE SAME TIME (not sequentially)
analysis = RunnableParallel(
    pros=pros_prompt | llm | StrOutputParser(),       # chain 1
    cons=cons_prompt | llm | StrOutputParser(),       # chain 2 (runs in parallel)
    use_cases=usecase_prompt | llm | StrOutputParser() # chain 3 (runs in parallel)
)

# Invoke — all 3 run at once, results come back as a dict
result = analysis.invoke({"technology": "LangChain"})

# result = {
#   "pros": "1. Easy to chain... 2. ...",
#   "cons": "1. Steep learning curve... 2. ...",
#   "use_cases": "1. Chatbots... 2. ..."
# }

for section_name, content in result.items():
    print(f"\n{'='*40}")
    print(f"  {section_name.upper()}")
    print(f"{'='*40}")
    print(content)

### ✅ What Just Happened?

1. One input `{"technology": "LangChain"}` was sent in
2. Three separate chains ran **in parallel** — not one after another
3. Each chain's output was stored under its key: `pros`, `cons`, `use_cases`
4. The result is a single dict containing all 3 outputs

**Performance benefit:** If each chain takes 2 seconds, sequential = 6 seconds, parallel ≈ 2 seconds.

**When to use this:** Multi-angle analysis, generating multiple sections of a report, running the same prompt on different models for comparison.

---

## 3 · RunnableLambda — Inject Custom Python Logic Into a Chain

### The Problem
Not every step in your pipeline is an LLM call. You might need to:
- **Clean** the user's input (strip whitespace, lowercase)
- **Validate** data before sending to the LLM
- **Transform** the LLM output into a specific format
- **Log** intermediate results

### The Solution
`RunnableLambda` wraps **any Python function** so it works as a chain step.
Your function goes in, gets the previous step's output as input, and passes its return value to the next step.

In [ ]:
# --- Preprocessing: Clean user input before it hits the LLM ---

def clean_input(data: dict) -> dict:
    """Normalize user input: strip whitespace, lowercase, count words.
    
    This runs BEFORE the prompt template, so the LLM gets clean input.
    """
    raw_query = data["query"]
    cleaned = raw_query.strip().lower()  # remove extra spaces, normalize case
    
    print(f"  [Preprocessor] Raw:     '{raw_query}'")
    print(f"  [Preprocessor] Cleaned: '{cleaned}'")
    
    return {
        "query": cleaned,
        "word_count": len(cleaned.split()),  # useful metadata for downstream
    }


# --- Postprocessing: Structure the raw LLM output ---

def format_output(text: str) -> dict:
    """Take the raw string from the LLM and wrap it in a structured dict.
    
    This runs AFTER the output parser, so it receives a plain string.
    """
    return {
        "answer": text,
        "char_count": len(text),
        "preview": text[:80] + "..." if len(text) > 80 else text,
    }


# --- Build the full pipeline ---
# Flow: raw input → clean → prompt → LLM → parse → format

prompt = ChatPromptTemplate.from_template("Answer concisely: {query}")

chain = (
    RunnableLambda(clean_input)      # Step 1: preprocess (Python function)
    | prompt                          # Step 2: fill the prompt template
    | llm                             # Step 3: send to LLM
    | StrOutputParser()               # Step 4: extract string from response
    | RunnableLambda(format_output)   # Step 5: postprocess (Python function)
)

# Test with messy input — the preprocessor cleans it up
result = chain.invoke({"query": "  What is RLHF?  "})

print(f"\nFinal output:")
for key, value in result.items():
    print(f"  {key}: {value}")

### ✅ What Just Happened?

1. `clean_input` received the raw dict `{"query": "  What is RLHF?  "}`
2. It stripped whitespace and lowercased it → `{"query": "what is rlhf?", "word_count": 4}`
3. The clean dict went into the prompt template → `"Answer concisely: what is rlhf?"`
4. The LLM generated a response → raw string
5. `format_output` wrapped that string into a structured dict with metadata

**Key insight:** `RunnableLambda` is the bridge between Python logic and LLM chains. Any function that takes one argument and returns one value can be a chain step.

In [ ]:
# --- Alternative syntax: @chain decorator ---
# If your function already contains LLM calls, use @chain instead of RunnableLambda.
# It makes the function itself a Runnable (supports .invoke(), .batch(), .stream()).

from langchain_core.runnables import chain as chain_decorator

@chain_decorator
def enrich_and_answer(input_dict: dict) -> str:
    """Add domain context to the query, then ask the LLM."""
    query = input_dict["query"]
    
    # Add context that helps the LLM give a better answer
    enriched = f"The user is an ML engineer asking about AI/ML. Their question: {query}"
    
    # Call the LLM directly inside the function
    response = llm.invoke(enriched)
    return response.content

# Now it works like any other Runnable
print(enrich_and_answer.invoke({"query": "What is LoRA?"}))

---

## 4 · RunnableBranch — Route to Different Chains Based on Input

### The Problem
A coding question needs a different system prompt than a math question or a general question.
Using one generic prompt for everything gives worse results.

### The Solution
`RunnableBranch` acts like an **if/elif/else** for chains. It checks conditions on the input
and routes it to the appropriate specialized chain.

Think of it like a hospital triage desk — the nurse (router) checks your symptoms (input)
and sends you to the right specialist (chain).

In [ ]:
# --- Define specialized chains for each query type ---
# Each has a system prompt tailored to its domain

code_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Python expert. Give clear, working code with brief explanations."),
    ("human", "{query}")
])

math_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math tutor. Solve problems step by step, showing your work."),
    ("human", "{query}")
])

general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise and accurate."),
    ("human", "{query}")
])


# --- Build the router ---
# RunnableBranch takes a list of (condition, chain) tuples
# It checks conditions TOP TO BOTTOM — first match wins
# The last argument (no condition) is the DEFAULT fallback

router = RunnableBranch(
    # Condition 1: if query contains code-related keywords → use code chain
    (lambda x: any(kw in x["query"].lower() for kw in ["code", "python", "function", "class", "script"]),
     code_prompt | llm | StrOutputParser()),
    
    # Condition 2: if query contains math-related keywords → use math chain
    (lambda x: any(kw in x["query"].lower() for kw in ["calculate", "solve", "equation", "math", "integral"]),
     math_prompt | llm | StrOutputParser()),
    
    # Default: if no condition matches → use general chain
    general_prompt | llm | StrOutputParser(),
)

In [ ]:
# --- Test the router with 3 different query types ---

test_queries = [
    "Write a Python function to reverse a linked list",   # → code chain
    "Solve the equation 3x + 7 = 22",                     # → math chain
    "What is the capital of Japan?",                       # → general chain (default)
]

for query in test_queries:
    # Detect which chain will handle this (for visibility)
    if any(kw in query.lower() for kw in ["code", "python", "function", "class", "script"]):
        routed_to = "🐍 CODE chain"
    elif any(kw in query.lower() for kw in ["calculate", "solve", "equation", "math"]):
        routed_to = "🔢 MATH chain"
    else:
        routed_to = "💬 GENERAL chain (default)"
    
    result = router.invoke({"query": query})
    
    print(f"\nQuery:      {query}")
    print(f"Routed to:  {routed_to}")
    print(f"Answer:     {result[:120]}...")
    print("-" * 60)

### ✅ What Just Happened?

1. Each query entered the `RunnableBranch`
2. The branch checked conditions **top to bottom** — first match wins
3. The matching chain processed the query with its specialized system prompt
4. Queries that matched no condition went to the **default** (general chain)

**In production**, you'd use an LLM-based classifier instead of keyword matching for more accurate routing. But the pattern is the same.

---

## 5 · Fallbacks — Automatic Backup When a Model Fails

### The Problem
In production, models go down — rate limits, timeouts, API outages.
If your app only uses one model, a single failure = total downtime.

### The Solution
`.with_fallbacks()` lets you define backup models. If the primary fails,
LangChain **automatically retries** with the next model in the list.

Your user never sees the failure — they just get an answer (maybe from a different model).

In [ ]:
# --- Basic Fallback: GPT primary, Claude backup ---

primary_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
backup_model = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)

# If primary fails → automatically try backup
# You can chain multiple fallbacks: primary → backup1 → backup2
resilient_llm = primary_model.with_fallbacks([backup_model])

prompt = ChatPromptTemplate.from_template("Explain {concept} in 2 sentences.")
chain = prompt | resilient_llm | StrOutputParser()

# This will use GPT (primary) — if it works, Claude is never called
print("With working primary model:")
print(chain.invoke({"concept": "attention mechanism"}))

In [ ]:
# --- Simulate a failure to see fallback in action ---

# This model doesn't exist → will fail immediately
broken_model = ChatOpenAI(model="gpt-nonexistent-model", temperature=0)

# Attach Claude as the fallback
safe_llm = broken_model.with_fallbacks([backup_model])

chain = prompt | safe_llm | StrOutputParser()

# GPT fails → Claude catches it → user still gets an answer
print("With broken primary model (fallback kicks in):")
print(chain.invoke({"concept": "RLHF"}))

# The user never knows GPT failed — they just see the answer from Claude

### ✅ What Just Happened?

1. We created a model (`gpt-nonexistent-model`) that's guaranteed to fail
2. `.with_fallbacks([backup_model])` wrapped it with Claude as a safety net
3. When GPT threw an error, LangChain **caught it silently** and retried with Claude
4. The chain returned Claude's answer as if nothing went wrong

**Production tip:** You can stack multiple fallbacks: `primary.with_fallbacks([backup1, backup2, backup3])`. LangChain tries each one in order until one succeeds.

---

## 6 · Putting It All Together — A Real Pipeline

Now let's combine everything into one pipeline:
1. **Preprocess** the input (RunnableLambda)
2. **Classify** the query type (Python logic)
3. **Route** to the right chain (RunnableBranch)
4. **Return** a structured response

In [ ]:
# --- Step 1: Preprocessing + Classification ---

def classify_and_clean(data: dict) -> dict:
    """Clean the query and add a 'category' field for routing.
    
    In production, you'd use an LLM or ML classifier here.
    We're using keyword matching for simplicity.
    """
    query = data["query"].strip()
    
    # Simple keyword-based classification
    if any(kw in query.lower() for kw in ["compare", "vs", "versus", "difference"]):
        category = "comparison"
    elif any(kw in query.lower() for kw in ["how to", "tutorial", "build", "create", "steps"]):
        category = "tutorial"
    else:
        category = "general"
    
    print(f"  [Classifier] Query: '{query}' → Category: {category}")
    
    return {"query": query, "category": category}


# --- Step 2: Specialized chains for each category ---

comparison_chain = (
    ChatPromptTemplate.from_template(
        "Compare the following in a structured way with pros/cons for each side. "
        "Be concise — max 5 bullet points total:\n\n{query}"
    ) | llm | StrOutputParser()
)

tutorial_chain = (
    ChatPromptTemplate.from_template(
        "Give a brief step-by-step guide (max 5 steps). "
        "Include one code snippet if relevant:\n\n{query}"
    ) | llm | StrOutputParser()
)

general_chain = (
    ChatPromptTemplate.from_template(
        "Answer this concisely in 2-3 sentences:\n\n{query}"
    ) | llm | StrOutputParser()
)


# --- Step 3: Route based on classification ---

router = RunnableBranch(
    (lambda x: x["category"] == "comparison", comparison_chain),
    (lambda x: x["category"] == "tutorial", tutorial_chain),
    general_chain,  # default fallback
)


# --- Step 4: Full pipeline ---
# clean + classify → route → answer

full_pipeline = RunnableLambda(classify_and_clean) | router

In [ ]:
# --- Test with different query types ---

test_queries = [
    "Compare FAISS vs ChromaDB for vector search",   # → comparison chain
    "How to build a RAG pipeline in LangChain",       # → tutorial chain
    "What is an embedding model?",                    # → general chain
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"  QUERY: {query}")
    print(f"{'='*60}")
    
    result = full_pipeline.invoke({"query": query})
    print(f"\n{result}")

### ✅ What Just Happened?

1. Each query went through `classify_and_clean` → got cleaned + tagged with a category
2. `RunnableBranch` checked the `category` field and routed to the right chain
3. Each chain used a specialized prompt optimized for that query type
4. The user got a better answer because the right expert handled their question

This is the pattern behind most production LLM apps — preprocess, classify, route, respond.

---

## Key Takeaways

| Runnable | What It Does | Real-World Analogy |
|----------|-------------|--------------------|
| `RunnablePassthrough` | Forward input unchanged | Conveyor belt — package stays on it |
| `.assign()` | Add new fields while keeping existing ones | Sticking a new label on the package |
| `RunnableParallel` | Run multiple chains simultaneously | 3 analysts working on same question |
| `RunnableLambda` | Wrap any Python function as a chain step | Custom station on the assembly line |
| `RunnableBranch` | Route input to different chains by condition | Hospital triage desk |
| `.with_fallbacks()` | Automatic backup when model fails | Backup generator for power outages |

**Next →** [03 · Output Parsers](../03-output-parsers/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*